### Carga de Dependencias

In [6]:
import pandas as pd
import sys
sys.path.append('..')  # Si estás en notebooks/

from src.data.data_quality import analyze_data_quality, show_categorical_columns    

### Carga de datos y variables

In [12]:
df_train = pd.read_parquet("C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_train.parquet", engine="pyarrow")
df_test = pd.read_parquet("C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_test.parquet", engine="pyarrow")
df_oot = pd.read_parquet("C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_oot.parquet", engine="pyarrow")
df_oos = pd.read_parquet("C:/Users/carlo/OneDrive/Documentos/repos/meli-ds/data/processed/df_oos.parquet", engine="pyarrow")

In [3]:
varss = ['attribute4_change_rate',
 'attribute2',
 'attribute8',
 'attribute1_lag_7_max',
 'attribute1_lag_120_max',
 'attribute1_lag_120_mean',
 'attribute5_lag_15_mean',
 'attribute5_lag_60_mean',
 'attribute6_lag_15_std',
 'attribute6_lag_30_min',
 'attribute6_lag_30_std',
 'attribute6_lag_60_min',
 'attribute6_lag_90_std',
 'attribute2_change_rate',
 'attribute2_cumulative_change',
 'attribute4_cumulative_change']

In [ ]:
import numpy as np

cols_bin = [
    "attribute2_change_rate",
    "attribute2_cumulative_change",
    "attribute4_change_rate",
    "attribute4_cumulative_change"
]

def binarizar_columnas(df, columnas):
    df = df.copy()
    for col in columnas:
        if col in df.columns:
            df[f"{col}_bin"] = np.where(df[col].fillna(0) != 0, 1, 0)
    return df

# Aplicar en todos los datasets
df_train = binarizar_columnas(df_train, cols_bin)
df_test  = binarizar_columnas(df_test, cols_bin)
df_oot   = binarizar_columnas(df_oot, cols_bin)
df_oos   = binarizar_columnas(df_oos, cols_bin)



In [17]:
varss = ['attribute2_change_rate_bin',  
'attribute2_cumulative_change_bin',  
'attribute4_change_rate_bin',  
'attribute4_cumulative_change_bin',
 'attribute2',
 'attribute8',
 'attribute1_lag_7_max',
 'attribute1_lag_120_max',
 'attribute1_lag_120_mean',
 'attribute5_lag_15_mean',
 'attribute5_lag_60_mean',
 'attribute6_lag_15_std',
 'attribute6_lag_30_min',
 'attribute6_lag_30_std',
 'attribute6_lag_60_min',
 'attribute6_lag_90_std'
]

In [18]:
df_train.head()

,date,device,failure,attribute1,attribute2,attribute3,attribute4,attribute5,attribute6,attribute7,...,attribute7_cumulative_change,attribute8_change_rate,attribute8_cumulative_change,attribute9_change_rate,attribute9_cumulative_change,date_trunc,attribute2_change_rate_bin,attribute2_cumulative_change_bin,attribute4_change_rate_bin,attribute4_cumulative_change_bin
0,2015-01-01,S1F01085,0,215630672,56,0,52,6,407438,0,...,NaN,NaN,NaN,NaN,NaN,2015-01,0,0,0,0
1163,2015-01-02,S1F01085,0,1650864,56,0,52,6,407438,0,...,NaN,NaN,NaN,0.0,0.0,2015-01,0,0,0,0
2326,2015-01-03,S1F01085,0,124017368,56,0,52,6,407438,0,...,NaN,NaN,NaN,0.0,0.0,2015-01,0,0,0,0
3489,2015-01-04,S1F01085,0,128073224,56,0,52,6,407439,0,...,NaN,NaN,NaN,0.0,0.0,2015-01,0,0,0,0
4651,2015-01-05,S1F01085,0,97393448,56,0,52,6,408114,0,...,NaN,NaN,NaN,0.0,0.0,2015-01,0,0,0,0


### División de muestras

In [19]:
x_train = df_train[varss].copy()
y_train = df_train['failure'].copy()

x_test = df_test[varss]
y_test = df_test['failure']

x_oot = df_oot[varss].copy()
y_oot = df_oot['failure'].copy()

x_oos = df_oos[varss].copy()
y_oos = df_oos['failure'].copy()
    

In [20]:

quality_report = analyze_data_quality(x_train)
print(quality_report)

categorical_report = show_categorical_columns(x_train)
if categorical_report is not None:
    print(categorical_report)

                             Columna      Tipo  Porcentaje_Nulos  \
0         attribute2_change_rate_bin  Numérica               0.0   
1   attribute2_cumulative_change_bin  Numérica               0.0   
2         attribute4_change_rate_bin  Numérica               0.0   
3   attribute4_cumulative_change_bin  Numérica               0.0   
4                         attribute2  Numérica               0.0   
5                         attribute8  Numérica               0.0   
6               attribute1_lag_7_max  Numérica               0.0   
7             attribute1_lag_120_max  Numérica               0.0   
8            attribute1_lag_120_mean  Numérica               0.0   
9             attribute5_lag_15_mean  Numérica               0.0   
10            attribute5_lag_60_mean  Numérica               0.0   
11             attribute6_lag_15_std  Numérica               0.0   
12             attribute6_lag_30_min  Numérica               0.0   
13             attribute6_lag_30_std  Numérica  

### Optimización hiperparámetros

In [5]:
!pip install optuna

  Using cached colorlog-6.9.0-py3-none-any.whl.metadata (10 kB)
   ---------------------------------------- 0.0/400.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/400.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/400.9 kB ? eta -:--:--
   --- ----------------------------------- 41.0/400.9 kB 279.3 kB/s eta 0:00:02
   ---------- --------------------------- 112.6/400.9 kB 652.2 kB/s eta 0:00:01
   ----------- -------------------------- 122.9/400.9 kB 654.9 kB/s eta 0:00:01
   ----------------------- ---------------- 235.5/400.9 kB 1.0 MB/s eta 0:00:01
   ---------------------------------------  399.4/400.9 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 400.9/400.9 kB 1.3 MB/s eta 0:00:00
Using cached colorlog-6.9.0-py3-none-any.whl (11 kB)



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
pip install imbalanced-learn

   ---------------------------------------- 0.0/240.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/240.0 kB ? eta -:--:--
   - -------------------------------------- 10.2/240.0 kB ? eta -:--:--
   - -------------------------------------- 10.2/240.0 kB ? eta -:--:--
   ------ -------------------------------- 41.0/240.0 kB 281.8 kB/s eta 0:00:01
   ----------------- -------------------- 112.6/240.0 kB 731.4 kB/s eta 0:00:01
   ---------------------------------------  235.5/240.0 kB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 240.0/240.0 kB 1.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
# Requisitos (instala si hace falta):
# pip install optuna lightgbm catboost scikit-learn imbalanced-learn

import optuna
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer, average_precision_score
from sklearn.ensemble import HistGradientBoostingClassifier

import lightgbm as lgb
from catboost import CatBoostClassifier

from imblearn.combine import SMOTETomek

# ----------------------------------------------------------
# Config: ajusta según prefieras
# ----------------------------------------------------------
N_TRIALS = 100            # número de trials por modelo (ajusta)
N_SPLITS = 5             # folds CV
RANDOM_STATE = 42

# Asume que ya tienes:
# x_train, y_train  (pandas DataFrame / array)
# Si no, carga tus datos antes de ejecutar.

# ----------------------------------------------------------
# 0) Rebalanceo: SMOTE + Tomek (solo en train)
# ----------------------------------------------------------
print("Aplicando SMOTE + Tomek sobre x_train (solo entrenamiento)...")
smote_tomek = SMOTETomek(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote_tomek.fit_resample(x_train, y_train)
print("Tamaños después de rebalanceo:", np.bincount(y_train_res.astype(int)))

# ----------------------------------------------------------
# Función helper: CV scoring (Average Precision)
# ----------------------------------------------------------
ap_scorer = make_scorer(average_precision_score, needs_proba=True)

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# ----------------------------------------------------------
# 1) LightGBM Optuna objective
# ----------------------------------------------------------
def objective_lgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 10.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 50.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 50.0),
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    }
    model = lgb.LGBMClassifier(**params)
    scores = cross_val_score(model, X_train_res, y_train_res, cv=cv, scoring="average_precision", n_jobs=-1)
    return scores.mean()

study_lgb = optuna.create_study(direction="maximize", study_name="LGBM_PR")
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS, show_progress_bar=True)

print("\nLightGBM - Mejor PR-AUC (cv):", study_lgb.best_value)
print("LightGBM - Mejores params:", study_lgb.best_params)

# Entrenar modelo final con mejores params sobre X_train_res
best_lgb = lgb.LGBMClassifier(**study_lgb.best_params, random_state=RANDOM_STATE, n_jobs=-1)
best_lgb.fit(X_train_res, y_train_res)

# ----------------------------------------------------------
# 2) CatBoost Optuna objective
# ----------------------------------------------------------
def objective_cat(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "depth": trial.suggest_int("depth", 3, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 50.0, log=True),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 0.0, 10.0),
        "verbose": 0,
        "random_state": RANDOM_STATE,
    }
    model = CatBoostClassifier(**params)
    # cross_val_score works with CatBoost if .fit supports sample_weight; uses predict_proba via scoring
    scores = cross_val_score(model, X_train_res, y_train_res, cv=cv, scoring="average_precision", n_jobs=-1)
    return scores.mean()

study_cat = optuna.create_study(direction="maximize", study_name="CatBoost_PR")
study_cat.optimize(objective_cat, n_trials=N_TRIALS, show_progress_bar=True)

print("\nCatBoost - Mejor PR-AUC (cv):", study_cat.best_value)
print("CatBoost - Mejores params:", study_cat.best_params)

best_cat = CatBoostClassifier(**study_cat.best_params, verbose=0, random_state=RANDOM_STATE)
best_cat.fit(X_train_res, y_train_res)

# ----------------------------------------------------------
# 3) HistGradientBoostingClassifier (sklearn) Optuna objective
# ----------------------------------------------------------
def objective_hgb(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 255),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
        "l2_regularization": trial.suggest_float("l2_regularization", 0.0, 50.0),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "random_state": RANDOM_STATE,
    }
    model = HistGradientBoostingClassifier(**params)
    scores = cross_val_score(model, X_train_res, y_train_res, cv=cv, scoring="average_precision", n_jobs=-1)
    return scores.mean()

study_hgb = optuna.create_study(direction="maximize", study_name="HGB_PR")
study_hgb.optimize(objective_hgb, n_trials=N_TRIALS, show_progress_bar=True)

print("\nHistGB - Mejor PR-AUC (cv):", study_hgb.best_value)
print("HistGB - Mejores params:", study_hgb.best_params)

best_hgb = HistGradientBoostingClassifier(**study_hgb.best_params, random_state=RANDOM_STATE)
best_hgb.fit(X_train_res, y_train_res)

# ----------------------------------------------------------
# 4) Resumen final: mejores modelos y scores en entrenamiento rebalanceado
# ----------------------------------------------------------
print("\n--- RESUMEN FINAL ---")
results = [
    ("LightGBM", best_lgb, study_lgb.best_value),
    ("CatBoost", best_cat, study_cat.best_value),
    ("HistGB", best_hgb, study_hgb.best_value)
]

for name, model, best_cv in results:
    print(f"\nModelo: {name}")
    print(f"  Mejor PR-AUC (CV en train rebalanceado): {best_cv:.6f}")
    # Opcional: obtener PR-AUC en x_train original (no rebalanceado) para referencia:
    try:
        proba_train_orig = model.predict_proba(x_train)[:, 1]
        ap_train_orig = average_precision_score(y_train, proba_train_orig)
        print(f"  PR-AUC en x_train (no rebalanceado): {ap_train_orig:.6f}")
    except Exception as e:
        print("  No se pudo calcular PR-AUC en train original:", e)

# ----------------------------------------------------------
# 5) Guardar los mejores parametros (opcional)
# ----------------------------------------------------------
best_params_summary = {
    "lgbm": study_lgb.best_params,
    "catboost": study_cat.best_params,
    "histgb": study_hgb.best_params
}
# guardarlo a disco si quieres
pd.Series(best_params_summary).to_json("best_params_models.json")
print("\nMejores hiperparámetros guardados en best_params_models.json")


Aplicando SMOTE + Tomek sobre x_train (solo entrenamiento)...


[I 2025-10-07 20:21:44,542] A new study created in memory with name: LGBM_PR


Tamaños después de rebalanceo: [81073 81073]


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-10-07 20:22:15,418] Trial 0 finished with value: 0.9991377885387024 and parameters: {'n_estimators': 455, 'learning_rate': 0.010600105566715146, 'num_leaves': 119, 'max_depth': 7, 'min_child_samples': 13, 'min_child_weight': 1.8096942749053642, 'subsample': 0.819626096366614, 'colsample_bytree': 0.7573472317254302, 'reg_alpha': 6.212166738067287, 'reg_lambda': 16.48036288014037}. Best is trial 0 with value: 0.9991377885387024.
[I 2025-10-07 20:22:25,701] Trial 1 finished with value: 0.9985158189808377 and parameters: {'n_estimators': 181, 'learning_rate': 0.060985447075301284, 'num_leaves': 239, 'max_depth': 4, 'min_child_samples': 70, 'min_child_weight': 0.03895710713708778, 'subsample': 0.6085946097366262, 'colsample_bytree': 0.4856187888578452, 'reg_alpha': 7.690256650579874, 'reg_lambda': 31.23512331969655}. Best is trial 0 with value: 0.9991377885387024.
[I 2025-10-07 20:22:38,475] Trial 2 finished with value: 0.999251800002946 and parameters: {'n_estimators': 315, 'learni

...

  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-10-07 21:00:12,465] Trial 0 finished with value: 0.9998806708728847 and parameters: {'iterations': 642, 'learning_rate': 0.03757240103616901, 'depth': 9, 'l2_leaf_reg': 0.005846558177730984, 'border_count': 37, 'bagging_temperature': 0.07373285055534495, 'random_strength': 3.774707998062757}. Best is trial 0 with value: 0.9998806708728847.
[I 2025-10-07 21:01:49,021] Trial 1 finished with value: 0.9997424365634316 and parameters: {'iterations': 1197, 'learning_rate': 0.18596032004359422, 'depth': 6, 'l2_leaf_reg': 0.0325916211417939, 'border_count': 164, 'bagging_temperature': 0.2579557444392573, 'random_strength': 0.7577110268487808}. Best is trial 0 with value: 0.9998806708728847.
[I 2025-10-07 21:03:47,358] Trial 2 finished with value: 0.9999572455304145 and parameters: {'iterations': 1946, 'learning_rate': 0.08398059774622744, 'depth': 3, 'l2_leaf_reg': 0.012168675903801031, 'border_count': 179, 'bagging_temperature': 0.11214132054540671, 'random_strength': 1.01868361304792

c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
2 fits failed out of a total of 5.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
2 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\carlo\OneDrive\Documentos\repos\meli-ds\venv\Lib\site-packages\catboost\core.py", line 5245, in fit
    self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None, None, None, None, baseline, use_bes

[W 2025-10-07 23:14:48,415] Trial 87 failed with parameters: {'iterations': 823, 'learning_rate': 0.1844991644504036, 'depth': 9, 'l2_leaf_reg': 0.46500970631720434, 'border_count': 198, 'bagging_temperature': 0.21008623087273576, 'random_strength': 4.723729291705562} because of the following error: The value nan is not acceptable.
[W 2025-10-07 23:14:48,417] Trial 87 failed with value nan.
[I 2025-10-07 23:15:54,834] Trial 88 finished with value: 0.9999616479447748 and parameters: {'iterations': 855, 'learning_rate': 0.18068511518075092, 'depth': 4, 'l2_leaf_reg': 0.07743705681898475, 'border_count': 94, 'bagging_temperature': 0.19915243149206108, 'random_strength': 4.718704516597033}. Best is trial 31 with value: 0.9999749788317566.
[I 2025-10-07 23:17:44,799] Trial 89 finished with value: 0.9999386920070444 and parameters: {'iterations': 1089, 'learning_rate': 0.03372486806483475, 'depth': 5, 'l2_leaf_reg': 0.4403726915738344, 'border_count': 200, 'bagging_temperature': 0.1325255606

KeyboardInterrupt: 